# Урок 12. Рекурсия

9 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 11](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-11.ipynb) · [Урок 13 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-13.ipynb)

---

Функция, вызывающая саму себя. Базовый случай и шаг рекурсии. Факториал, числа Фибоначчи, обход дерева. Глубина рекурсии.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-12", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Матрёшка внутри матрёшки

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g09/matryoshka.jpg" width="300" alt="Первая русская матрёшка, конец XIX века">

*Первая русская матрёшка, конец XIX века*

<sub>резьба В. Звёздочкина, роспись С. Малютина; фото RK812 · общественное достояние · Wikimedia Commons</sub>

Как открыть матрёшку? «Снять верх; если внутри есть ещё одна матрёшка —
открыть её тем же способом». Описание короткое, а работает для любого
числа кукол. Оно ссылается само на себя — и это не ошибка.

**Рекурсия** — это когда функция вызывает саму себя.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g09/droste.jpg" width="200" alt="Какао Droste: на банке нарисована женщина с банкой какао Droste">

*Какао Droste: на банке нарисована женщина с банкой какао Droste*

<sub>Alf van Beem · CC0 · Wikimedia Commons</sub>

Художники любят тот же приём. На банке голландского какао Droste
нарисована женщина, которая держит поднос с банкой какао Droste,
на которой нарисована та же женщина. В реальности картинка обрывается —
краска не бесконечна. В программе обрыв тоже обязателен, иначе всё
зависнет.

### Два обязательных куска

У любой рекурсивной функции есть:

* **базовый случай** — когда задача уже решена и вызывать себя не нужно.
  Самая маленькая матрёшка, которая не открывается;
* **шаг рекурсии** — сведение задачи к такой же, но меньшей, и вызов
  самой себя.

Забыли базовый случай — функция будет вызывать себя вечно. Python
не даст ей сломать компьютер: после примерно тысячи вложенных вызовов
он остановит программу с ошибкой `RecursionError`.

### Факториал

Факториал числа n (записывают `n!`) — произведение всех чисел от 1 до n.
Например, `5! = 1 · 2 · 3 · 4 · 5 = 120`.

Заметим главное: `5! = 5 · 4!`. То есть факториал числа выражается
через факториал меньшего числа. А `1!` равен 1 — вот и базовый случай.

```
факториал(n) = 1,                если n ≤ 1
факториал(n) = n · факториал(n−1), иначе
```

### Как это выполняется

Каждый вызов ждёт, пока закончится вложенный. Вызовы складываются
в стопку (её так и называют — **стек вызовов**), а потом разбираются
в обратном порядке:

```
  факториал(4)
    4 · факториал(3)
        3 · факториал(2)
            2 · факториал(1)
                1                 ← базовый случай, пошли обратно
            2 · 1 = 2
        3 · 2 = 6
    4 · 6 = 24
```

Сначала «вглубь» до базового случая, потом «обратно» с умножениями.
Поэтому у рекурсии есть цена: каждый незавершённый вызов занимает
память.

## Смотрим, как это работает

### Пример 1. Факториал двумя способами

In [ ]:
def факториал(n):
    if n <= 1:
        return 1
    return n * факториал(n - 1)


def факториал_циклом(n):
    результат = 1
    for i in range(2, n + 1):
        результат *= i
    return результат


print(факториал(5), факториал_циклом(5))
print(факториал(10), факториал_циклом(10))

Результаты одинаковые. Рекурсивная версия короче и ближе к формуле,
версия с циклом экономнее. Любую рекурсию можно переписать циклом,
но иногда цикл получается заметно сложнее — например, при обходе дерева.

### Пример 2. Видим порядок вызовов

In [ ]:
def факториал_вслух(n, глубина=0):
    отступ = "    " * глубина
    print(f"{отступ}вызов факториал({n})")
    if n <= 1:
        print(f"{отступ}базовый случай → 1")
        return 1
    результат = n * факториал_вслух(n - 1, глубина + 1)
    print(f"{отступ}вернули {результат}")
    return результат


факториал_вслух(4)

Отступы показывают глубину: сначала спуск, потом подъём с ответами.

### Пример 3. Числа Фибоначчи

Каждое число — сумма двух предыдущих: 1, 1, 2, 3, 5, 8, 13, 21…
Базовых случаев здесь два: первое и второе числа.

In [ ]:
def фибоначчи(n):
    if n <= 2:
        return 1
    return фибоначчи(n - 1) + фибоначчи(n - 2)


for i in range(1, 11):
    print(фибоначчи(i), end=" ")
print()

Красиво, но крайне расточительно: `фибоначчи(30)` считает
`фибоначчи(10)` тысячи раз заново. Посчитаем вызовы.

In [ ]:
вызовов = 0


def фибоначчи_со_счётчиком(n):
    global вызовов
    вызовов += 1
    if n <= 2:
        return 1
    return фибоначчи_со_счётчиком(n - 1) + фибоначчи_со_счётчиком(n - 2)


for n in (10, 20, 25):
    вызовов = 0
    значение = фибоначчи_со_счётчиком(n)
    print(f"фибоначчи({n}) = {значение}, вызовов: {вызовов}")

Число вызовов растёт примерно вдвое с каждым шагом. В 11 классе
вы научитесь лечить это запоминанием уже посчитанного —
приём называется мемоизацией.

### Пример 4. Ханойская башня

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g09/hanoy.jpg" width="260" alt="Ханойская башня">

*Ханойская башня*

<sub>автор не указан · CC BY-SA 3.0 · Wikimedia Commons</sub>

Головоломку придумал французский математик Эдуар Люка в 1883 году.
Нужно перенести стопку дисков с одного стержня на другой, беря
по одному диску и никогда не кладя больший на меньший.

Рекурсивное решение умещается в три строки. Чтобы перенести n дисков
с A на C: перенести верхние n−1 дисков на B, переложить самый большой
на C, перенести n−1 дисков с B на C.

In [ ]:
def ханой(n, откуда, куда, через, ходы):
    if n == 0:
        return ходы
    ханой(n - 1, откуда, через, куда, ходы)
    ходы.append(f"{откуда}→{куда}")
    ханой(n - 1, через, куда, откуда, ходы)
    return ходы


ходы = ханой(3, "A", "C", "B", [])
print(len(ходы), "хода:", " ".join(ходы))

for дисков in range(1, 11):
    print(дисков, "дисков —", len(ханой(дисков, "A", "C", "B", [])), "ходов")

Число ходов равно 2ⁿ − 1. По легенде монахи перекладывают 64 диска,
и когда закончат, наступит конец света. При скорости один диск
в секунду им понадобится около 585 миллиардов лет.

## Пробуем сами

### Задача 1. Как называется

Как называется часть рекурсивной функции, где она перестаёт вызывать
саму себя?

In [ ]:
#@title 🧩 Задача 1. Часть функции { display-mode: "form" }
#@markdown Выберите ответ
часть = "выбери ответ" #@param ["выбери ответ", "базовый случай", "шаг рекурсии", "стек вызовов"]

si.ответ("1", часть, "cf1c29e96ab00b73",
         hint="Самая маленькая матрёшка.")

### Задача 2. Сумма до n

Напишите **рекурсивную** функцию: сумма всех чисел от 1 до n.
`сумма(4)` → 10. Для n ≤ 0 возвращайте 0.

In [ ]:
def сумма(n):
    return ...

In [ ]:
si.check("2", сумма, [
    (4, 10),
    (1, 1),
    (0, 0),
    (10, 55),
    (100, 5050),
])

### Задача 3. Сколько вызовов

Функция `факториал` из примера 1 (базовый случай `n <= 1`).
Сколько всего вызовов произойдёт при `факториал(5)`, считая самый
первый?

In [ ]:
#@title 🧩 Задача 3. Сколько вызовов { display-mode: "form" }
#@markdown Впишите число
вызовов_ответ = 0 #@param {type:"integer"}

si.ответ("3", вызовов_ответ, "ef2d127de37b942b",
         hint="Загляните в вывод примера 2 и продолжите до пяти.")

### Задача 4. Степень числа

Рекурсивная функция возведения в степень: `степень(2, 10)` → 1024.
Показатель — целое число, не меньше нуля. Любое число в нулевой
степени равно 1 — вот базовый случай. Оператором `**` пользоваться
нельзя.

In [ ]:
def степень(основание, показатель):
    return ...

In [ ]:
si.check("4", степень, [
    ((2, 10), 1024),
    ((5, 0), 1),
    ((3, 4), 81),
    ((7, 1), 7),
])

### Задача 5. Ханойская башня

Сколько ходов нужно, чтобы перенести башню из 10 дисков?

In [ ]:
#@title 🧩 Задача 5. Десять дисков { display-mode: "form" }
#@markdown Впишите число
ходов_10 = 0 #@param {type:"integer"}

si.ответ("5", ходов_10, "6629ddae3736e894",
         hint="Посмотрите на таблицу из примера 4 или посчитайте 2¹⁰ − 1.")

### Задача 6. Сумма цифр числа

Рекурсивная функция суммы цифр натурального числа.
`сумма_цифр(1234)` → 10.

Подсказка: последняя цифра — это `n % 10`, а всё остальное — `n // 10`.

In [ ]:
def сумма_цифр(n):
    return ...

In [ ]:
si.check("6", сумма_цифр, [
    (1234, 10),
    (7, 7),
    (0, 0),
    (999, 27),
    (1000000, 1),
])

### Задача 7. Что будет без базового случая

```python
def плохая(n):
    return n + плохая(n - 1)

плохая(5)
```

In [ ]:
#@title 🧩 Задача 7. Без базового случая { display-mode: "form" }
#@markdown Выберите ответ
что_будет = "выбери ответ" #@param ["выбери ответ", "вернёт 0", "RecursionError — бесконечная рекурсия", "компьютер зависнет навсегда"]

si.ответ("7", что_будет, "f1c60684570cbec4",
         hint="Python считает глубину вызовов и вовремя останавливает программу.")

## Домашнее задание

### Домашнее задание 1. Количество цифр

Рекурсивная функция: сколько цифр в натуральном числе.
`цифр(1234)` → 4, `цифр(0)` → 1.

In [ ]:
def цифр(n):
    return ...

In [ ]:
si.check("дз1", цифр, [
    (1234, 4),
    (0, 1),
    (9, 1),
    (10, 2),
    (1000000, 7),
])

### Домашнее задание 2. Переворот строки

Рекурсивная функция, которая возвращает строку задом наперёд.
`наоборот("абвг")` → `"гвба"`. Срезом `[::-1]` пользоваться нельзя:
отрежьте первый символ и переверните остаток.

In [ ]:
def наоборот(строка):
    return ...

In [ ]:
si.check("дз2", наоборот, [
    ("абвг", "гвба"),
    ("", ""),
    ("я", "я"),
    ("рекурсия", "яисрукер"),
])

### Домашнее задание 3. Матрёшка в тетради

Нарисуйте дерево вызовов для `фибоначчи(5)`: от корня вниз, каждый
вызов порождает два. Посчитайте по рисунку, сколько всего вызовов,
и сколько раз считается `фибоначчи(2)`. Потом проверьте счётчиком
из примера 3.

---

### Любопытно

В словаре программистов есть шутка: «Рекурсия — см. рекурсия».
Google на запрос «recursion» много лет отвечал «Возможно, вы имели
в виду: recursion».

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 11](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-11.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 13 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-13.ipynb)